# Tree2SQL Performance Benchmarks

Detailed performance analysis comparing Python UDF vs native SQL CASE expression execution in DuckDB across:
- Multiple tree depths (3, 5, 7, 10)
- Full Bank Marketing dataset (45,211 rows)
- 5 timed runs each (averages reported)


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import time

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from tree2sql import TreeToSQL, Database, Benchmark
from data.load_dataset import load_bank_marketing

print("Imports OK ✓")


In [ ]:
X_train, X_test, y_train, y_test, feature_names, full_df = load_bank_marketing()

db = Database("bench_tree2sql.duckdb")
X_full = full_df.drop(columns=["y"])
db.load_dataframe(X_full, "bank_data", if_exists="replace")
print(f"Loaded {db.row_count('bank_data'):,} rows into DuckDB ✓")


In [ ]:
bench = Benchmark(db, table_name="bank_data", n_runs=5)

print("=" * 55)
print("Depth-sweep benchmark (classification, 5 runs each)")
print("=" * 55)
results = bench.run_depth_sweep(
    X_train, y_train,
    feature_columns=feature_names,
    depths=[3, 5, 7, 10],
    task="classification",
)


In [ ]:
summary = bench.summary_dataframe(results)
print("\n=== Benchmark Summary ===")
print(summary.to_string(index=False))
print(f"\nBest speedup: {summary['speedup_x'].max():.1f}×  "
      f"(depth {summary.loc[summary['speedup_x'].idxmax(), 'max_depth']})")


In [ ]:
fig = bench.plot_speedup(results, save_path="benchmark_chart.png")
plt.show()
print("Chart saved to benchmark_chart.png")


In [ ]:
# ── Additional chart: per-run variability (box/violin) ───────────────────────
fig2, ax = plt.subplots(figsize=(10, 5))
fig2.suptitle("Execution Time Distribution per Tree Depth (5 runs)", fontsize=12)

depths = [r.max_depth for r in results]
x = np.arange(len(depths))
width = 0.35

for i, r in enumerate(results):
    ax.scatter([x[i] - width/2] * len(r.udf_times), r.udf_times,
               color="#e74c3c", alpha=0.7, s=50, zorder=3)
    ax.scatter([x[i] + width/2] * len(r.sql_times), r.sql_times,
               color="#2ecc71", alpha=0.7, s=50, zorder=3)

# Means as horizontal bars
udf_means = [r.udf_mean for r in results]
sql_means = [r.sql_mean for r in results]
ax.bar(x - width/2, udf_means, width, label="UDF (mean)", color="#e74c3c", alpha=0.3)
ax.bar(x + width/2, sql_means, width, label="SQL (mean)", color="#2ecc71", alpha=0.3)

ax.set_xticks(x)
ax.set_xticklabels([f"depth={d}" for d in depths])
ax.set_ylabel("Execution Time (s)")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
fig2.savefig("benchmark_variability.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
db.close()
print("Database closed. Benchmark complete ✓")
